# Phase 2 — Claim Extraction, Verdict, and /verify

This notebook covers three pieces of **Phase 2**: turning raw report text into
a list of checkable claims, verifying each one against the Phase 1 knowledge
base, and wiring both into `POST /verify`. The Airflow DAG isn't built yet —
see [docs/phase-2-rag-pipeline.md](../docs/phase-2-rag-pipeline.md) for the plan.

```mermaid
flowchart LR
    TEXT[Report text]
    LLM[Claim Extractor LLM]
    CLAIMS["Claims: entity, metric, value, date"]
    RAG[hybrid_search retrieval]
    JUDGE[LLM-as-judge]
    VERDICT["Verdict: VERIFIED / REFUTED / INSUFFICIENT + quote"]
    API[POST /verify]

    TEXT --> LLM --> CLAIMS --> RAG --> JUDGE --> VERDICT --> API

    API -.->|next: not built yet| DAG[Airflow DAG]
```


In [1]:
import sys
sys.path.insert(0, "..")

## Claim extractor — structured output, not free-text parsing

`src/claim_extractor.py` asks the LLM for a typed `ClaimList` (via `ChatOpenAI(...).with_structured_output(ClaimList)`), not a text blob to regex apart. Each `Claim` has `text` (verbatim sentence), `entity`, `metric`, `value`, `date` — independently checkable against a source later.

The system prompt tells the model to skip opinions and forward-looking statements, keeping only claims with an entity + a number attached.

In [2]:
from src.claim_extractor import extract_claims

### Example: a snippet from a business report

Two checkable numeric claims (Apple revenue/net income, Walmart revenue), mixed with two opinion sentences that shouldn't survive extraction.

In [3]:
report_text = """Apple Inc. delivered another strong year. Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.
Management remains optimistic about the upcoming product cycle. Separately, Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31. 
Analysts believe the retail sector will keep growing."""

print(report_text)

Apple Inc. delivered another strong year. Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.
Management remains optimistic about the upcoming product cycle. Separately, Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31. 
Analysts believe the retail sector will keep growing.


In [4]:
claims = extract_claims(report_text)
for c in claims:
    print(c)

text='Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.' entity='Apple Inc.' metric='Revenue' value=416161000000.0 date='fiscal year ending 2025-09-27'
text='Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.' entity='Apple Inc.' metric='Net income' value=112010000000.0 date='fiscal year ending 2025-09-27'
text='Separately, Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31.' entity='Walmart' metric='Revenue' value=713163000000.0 date='fiscal year ending 2026-01-31'


Both opinion sentences ("Management remains optimistic...", "Analysts believe...") were correctly dropped — no entity+number attached. All three extracted claims have a clean `entity`/`metric`/`value`/`date`, ready to look up against the Phase 1 knowledge base.

**Rough edge:** the first two claims share the same `text` — the model split one sentence containing two numbers ("revenue... while net income...") into two claims but kept the whole sentence as `text` for both, instead of the specific clause. Doesn't break the structured fields (each is independently correct), but worth a tighter prompt if per-claim quoting matters later.

## Model choice: OpenRouter, with a local Ollama fallback

Default is OpenRouter's free tier (`LLM_PROVIDER=openrouter`, `LLM_MODEL=tencent/hy3:free`) — $0 cost, fits a capstone with no LLM budget.

Originally tried `nvidia/nemotron-3-ultra-550b-a55b:free`: every call failed with `DEGRADED function cannot be invoked` — confirmed with a bare `openai` client too, so it's an outage on the provider's side, not a bug here. Swapped to `tencent/hy3:free`, which works cleanly with structured output (see extraction above).

`src/config.py` also supports `LLM_PROVIDER=ollama` — routes to a local Ollama model (`ornith:latest`) via its OpenAI-compatible endpoint
(`http://localhost:11434/v1`, dummy API key). No API key, no rate limits, works offline. Useful as a fallback when the OpenRouter free tier is degraded or rate-limited — not as the default, since it's CPU-only on this machine (~20s/call once warm, 2+ min cold start).

In [5]:
from langchain_openai import ChatOpenAI
from src.claim_extractor import ClaimList, SYSTEM_PROMPT

ollama_llm = ChatOpenAI(
    model="ornith:latest",
    api_key="ollama",
    base_url="http://localhost:11434/v1",
    temperature=0,
).with_structured_output(ClaimList)

result = ollama_llm.invoke([("system", SYSTEM_PROMPT), ("user", report_text)])
for c in result.claims:
    print(c)

text='Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000' entity='Apple Inc.' metric='revenue' value=416161000000.0 date='fiscal year ending 2025-09-27'
text='net income came in at $112,010,000,000' entity='Apple Inc.' metric='net income' value=112010000000.0 date='fiscal year ending 2025-09-27'
text='Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31' entity='Walmart' metric='revenue' value=713163000000.0 date='fiscal year ending 2026-01-31'


Same three claims, correctly split, and this time each `text` is the exact clause ("Revenue for..." vs "net income came in at...") instead of the shared full sentence OpenRouter's `tencent/hy3:free` produced above — a nice reminder that "free tier" and "good enough" don't always point at the same model, and it's worth A/B-ing more than one before locking in a default.

## Verdict — claim + retrieved evidence -> VERIFIED / REFUTED / INSUFFICIENT

`src/verifier.py`'s `verify_claim(claim, prompt=VERDICT_PROMPT_V1)`:

1. Embeds the claim text, runs it through `hybrid_search` (RRF over pgvector
   + Postgres full-text, from Phase 1/3) against the knowledge base.
2. Passes the claim + top-5 retrieved chunks to an LLM-as-judge, which
   returns a structured `Verdict` (`verdict`, `source`, `quote`).

`prompt` is a plain function argument, not hardcoded — so Phase 3 can call
`verify_claim` with a second prompt variant on the same claims and compare
accuracy (the "LLM evaluation, 2 prompts" line in the capstone rubric).

In [1]:
from src.claim_extractor import Claim
from src.verifier import verify_claim

### Three cases, one from each `data/eval_claims.csv` bucket

- A `VERIFIED` claim (Apple revenue, correct value)
- A `REFUTED` claim (Apple revenue, wrong value — same source chunk as above)
- An `INSUFFICIENT` claim (Netflix — not in the knowledge base at all)

In [1]:
cases = [
    Claim(text="Apple's revenue for fiscal year ending 2025-09-27 was $416,161,000,000.",
          entity="Apple", metric="Revenue", value=416161000000.0, date="2025-09-27"),
    Claim(text="Apple's revenue for fiscal year ending 2025-09-27 was $250,000,000,000.",
          entity="Apple", metric="Revenue", value=250000000000.0, date="2025-09-27"),
    Claim(text="Netflix's revenue for fiscal year 2025 was $39 billion.",
          entity="Netflix", metric="Revenue", value=39000000000.0, date="2025"),
]

for claim in cases:
    v = verify_claim(claim)
    print(claim.text, "->", v.verdict, "|", v.source, "|", (v.quote or "")[:80])

Apple's revenue for fiscal year ending 2025-09-27 was $416,161,000,000. -> VERIFIED | Apple Inc. (AAPL) — Revenue | Apple Inc. (AAPL) reported Revenue of $416,161,000,000 for fiscal year ending 20
Apple's revenue for fiscal year ending 2025-09-27 was $250,000,000,000. -> REFUTED | Apple Inc. (AAPL) — Revenue | Apple Inc. (AAPL) reported Revenue of $416,161,000,000 for fiscal year ending 20
Netflix's revenue for fiscal year 2025 was $39 billion. -> INSUFFICIENT | None |


All three correct:

- **VERIFIED** — matched the exact chunk, same value.
- **REFUTED** — retrieval pulled the *same* Apple revenue chunk (it's the
  closest match by entity+metric), and the judge correctly compared values
  and caught the mismatch instead of just confirming "some Apple revenue
  chunk exists."
- **INSUFFICIENT** — Netflix isn't in the knowledge base (`TICKERS` in
  `fetch_secedgar.py` only covers 8 companies). Retrieval still returns its
  top-5 nearest chunks regardless — the judge had to actively recognize
  none of them cover Netflix, rather than defaulting to whatever ranked
  first. This is the prompt instruction doing its job, not an empty-results
  shortcut in the code.

## `POST /verify` — the two pieces wired into one endpoint

`src/api.py` chains `extract_claims` -> `verify_claim` per claim and returns
a `Verdict` list. Calling it in-process via FastAPI's `TestClient` (no need
to run a separate `uvicorn` process for this demo).

In [1]:
from fastapi.testclient import TestClient
from src.api import app

client = TestClient(app)

In [1]:
import json

resp = client.post("/verify", json={"text": "Apple reported revenue of $416,161,000,000 for fiscal year ending 2025-09-27."})
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

200
{
  "claims": [
    {
      "claim": "Apple reported revenue of $416,161,000,000 for fiscal year ending 2025-09-27.",
      "verdict": "VERIFIED",
      "source": "Apple Inc. (AAPL) \u2014 Revenue",
      "quote": "Apple Inc. (AAPL) reported Revenue of $416,161,000,000 for fiscal year ending 2025-09-27 (10-K filed 2025-10-31)."
    }
  ]
}


**On free-tier latency:** this single-claim call took ~56s end-to-end
(claim extraction + embedding + retrieval + LLM-as-judge). Multi-claim
reports take proportionally longer, and OpenRouter's free tier is not
latency-consistent — some calls finish in ~10s, others take much longer
under load. `src/llm.py` sets an explicit per-call `timeout` (30s on
OpenRouter, 180s on the Ollama fallback) with one retry, so a stuck request
fails fast instead of hanging — found this the hard way while building this
notebook: the default `openai` client timeout is 600s, so a slow call just
sat there silently until we added the explicit timeout.

## What's next

Claim extraction, verdict logic, and `POST /verify` are all working. Still
open (see [docs/phase-2-rag-pipeline.md](../docs/phase-2-rag-pipeline.md)):

- Airflow DAG for scheduled daily ingestion — the last Phase 2 item

[← Back to README](../README.md)